In [25]:
from  ..middleWare import *
#初始化模型
load_dotenv(override=True)

DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')
model=init_chat_model(
    model='deepseek-v4-flash',
    model_provider='deepseek',
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={'thinking':{"type":'disabled'}},
)
#定义工具
@tool
def get_news() ->str:
    '''
    查询当日新闻
    '''
    return f'今日伊朗袭击了美国'
@tool
def read_email(email_id:str):
    '''根据邮件id阅读邮件内容'''
    return f'id为{email_id}的邮件内容为空'
@tool(parse_docstring=True)
def send_email(fro:str,to:str,subject:str):
    '''
    发送邮件

    Args:
        fro:发送人
        to:收件人
        subject:邮件主题
    '''
    return f'{fro}发送了主题为{subject}的邮件到{to}'
#创建智能体使用人工审核中间件
#HumanInTheLoopMiddleware包含interrupt_on和description_prefix参数
#interrupt_on是InterruptOnConfig类，是TypeDict的子类，是字典类型
#键为工具名，值包括False和True以及自定义的allowed_decisions和description
#False时表示不中断工具调用，True表示表示所有决策(approve, edit, reject) 都可以选择
#allowed_decisions表示可以自定义三种决策选择，description为特定工具的中断描述信息，优先级高于description_prefix
#description_prefix会更改所有工具中断描述信息
myagent=create_agent(
    model=model,
    tools=[get_news,read_email,send_email],
    #创建检查点实例
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                'get_news':True,
                'read_email':False,
                'send_email':{
                    'allowed_decisions':['approve','reject'],
                    'description':'发送邮件中断...'
                },
            },
            description_prefix='工具调用中断...'
        )
    ],
)

In [28]:
#一个agent实例有一个checkpoint存储在内存，一个checkpoint有多个thread保存会话状态相互隔离
config={'configurable':{'thread_id':'1'}}
messages=[
    # HumanMessage('请帮我查询今日新闻，然后将新闻作为邮件主题从111@h.com发送到222@h.com'),
    HumanMessage('请帮我查询今日新闻，然后从111@h.com发送邮件主题为战争的邮件到222@h.com，同时做这两件事')
]
response=myagent.invoke(
    {
        'messages': messages,
    },
    config=config,
)
print('='*20+'原始响应'+'='*20+'\n')
rprint(response)
# interrupts=response.get('__interrupts__',[])
# rprint(interrupts)

====================原始响应====================



{
    'messages': [
        HumanMessage(
            content='请帮我查询今日新闻，然后从111@h.com发送邮件主题为战争的邮件到222@h.com，同时做这两件事',
            additional_kwargs={},
            response_metadata={},
            id='38d3b4b2-cf24-4741-bb24-65896e26310d'
        ),
        AIMessage(
            content='好的，我来同时执行这两个操作。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 105,
                    'prompt_tokens': 423,
                    'total_tokens': 528,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 39
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': 'ede9fcfa-1f4c-4276-b370-1f387872bcc3',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f88bf-850f-79b3-a7ca-d455a3e17806-0',
            tool_calls=[
                {'name': 'get_news', 'args': {}, 'id': 'call_00_eWQmnE1IjmvYUDwvEtLf2296', 'type': 'tool_call'},
                {
                    'name': 'send_email',
                    'args': {'fro': '111@h.com', 'to': '222@h.com', 'subject': '战争'},
                    'id': 'call_01_pTbG90fCwEC4tLWBVTRU0528',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 423,
                'output_tokens': 105,
                'total_tokens': 528,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_news',
                        'args': {},
                        'description': '工具调用中断...\n\nTool: get_news\nArgs: {}'
                    },
                    {
                        'name': 'send_email',
                        'args': {'fro': '111@h.com', 'to': '222@h.com', 'subject': '战争'},
                        'description': '发送邮件中断...'
                    }
                ],
                'review_configs': [
                    {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'send_email', 'allowed_decisions': ['approve', 'reject']}
                ]
            },
            id='a7e7917302bfd9bd894461aab30ee7f4'
        )
    ]
}

In [31]:
#中断后人工审批执行，Command中的resume用于中断后传入指令继续执行
from langgraph.types import Command
interrupts=response.get('__interrupt__',[])
action_requests=interrupts[0].value['action_requests']
decisions={
    'decisions':[]
}
get_news_decision={
    'type':'approve'
}
send_email_decision={
    'type':'approve'
}
for action_request in  action_requests:
    if action_request['name']=='get_news':
        decisions['decisions'].append(get_news_decision)
    if action_request['name']=='send_email':
        decisions['decisions'].append(send_email_decision)
#中断后，传入审批指令resume,还有update指令
if interrupts:
    resumed_response=myagent.invoke(
        Command(resume=decisions),
        config=config,
    )
    for msg in resumed_response["messages"]:
        msg.pretty_print()

================================ Human Message =================================

请帮我查询今日新闻，然后从111@h.com发送邮件主题为战争的邮件到222@h.com，同时做这两件事
================================== Ai Message ==================================

好的，我来同时执行这两个操作。
Tool Calls:
  get_news (call_00_eWQmnE1IjmvYUDwvEtLf2296)
 Call ID: call_00_eWQmnE1IjmvYUDwvEtLf2296
  Args:
  send_email (call_01_pTbG90fCwEC4tLWBVTRU0528)
 Call ID: call_01_pTbG90fCwEC4tLWBVTRU0528
  Args:
    fro: 111@h.com
    to: 222@h.com
    subject: 战争
================================= Tool Message =================================
Name: get_news

今日伊朗袭击了美国
================================= Tool Message =================================
Name: send_email

111@h.com发送了主题为战争的邮件到222@h.com
================================== Ai Message ==================================

两件事已经同时完成了！下面是结果汇总：

---

### ✅ 今日新闻
**伊朗袭击了美国** — 今日重大国际新闻。

### ✅ 邮件发送结果
- **发件人：** 111@h.com
- **收件人：** 222@h.com
- **主题：** 战争
- **状态：** 已成功发送

两件事已同步执行完毕！如果您还需要进一步操作，请随时告诉我。
